In [ ]:
import numpy as np
import polars as pl
from anngeno import AnnGeno

## Sanity check AnnGeno

In [ ]:
vm = pl.read_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/variant_metadata.parquet")
vm

In [ ]:
an = pl.read_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/annotations.parquet")
an['region'].unique()

In [ ]:
ag = AnnGeno(
    "/home/dnanexus/data_dir/dms_anngeno.ag",
    low_mem = True
)
ag

In [ ]:
gene_id = 'ENSG00000106633' # GCK
reg_dict = ag.get_region(gene_id)

geno = reg_dict['genotypes']
anno_df = reg_dict['annotations']
geno

In [ ]:
mac = np.load("/home/dnanexus/data_dir/all_mac.npy")
gene_mac = mac[anno_df['col']]
gene_mac

In [ ]:
geno_sum = geno.sum(axis=1)
geno_sum

In [ ]:
# Boolean mask where values differ
diff_mask = geno_sum != gene_mac

# Indices where they differ
diff_positions = np.where(diff_mask)[0]
diff_positions

## Create new subsetted AnnGeno

### Steps

1. first run `python subset_anngeno.py`
2. then create a dir `new.ag`, this will be the new anngeno
3. move the `new_genotypes/` into `new.ag/zarr_store/genotypes`
4. add the `samples/` zarr array to `new.ag/zarr_store/samples` as well
5. create `new.ag/zarr_store/zarr.json` and `new.ag/zarr_store/samples/zarr.json`

The code below will create the files:
- `new.ag/variant_metadata.parquet`
- `new.ag/annotations.parquet`

### Create metadata and annotations parquets

In [ ]:
ag = AnnGeno(
    "/home/dnanexus/data_dir/dms_anngeno.ag",
    # "/home/dnanexus/data_dir/dms_coding.ag",
    low_mem = True
)
ag

In [ ]:
# an = pl.scan_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/annotations.parquet")
an = pl.scan_parquet("/home/dnanexus/data_dir/dms_coding.ag/annotations.parquet")

coding_vars = an.filter((pl.col("relative_cds_position_is_nan") == 0)| (pl.col("loftee_hc_is_nan") == 0 )).select("id").collect()["id"].unique().to_list()
coding_vars

In [ ]:
# Correctly sorted variant metadata
vm_og = pl.read_parquet('/home/dnanexus/data_dir/variant_metadata.parquet')
vm_og

In [ ]:
vm = vm_og.filter(pl.col('id').is_in(coding_vars))
vm = vm.with_columns(pl.arange(0, pl.len(), eager=False).alias("col"))
vm

In [ ]:
from scipy.stats import beta

beta_weights = (1, 25)

maf_beta = beta.pdf(vm["maf"].to_numpy(), beta_weights[0], beta_weights[1])

maf_df = pl.DataFrame({
    "id": vm["id"],
    "col": vm["col"],
    "af_ukb": vm["maf"],
    "afbeta_ukb": maf_beta
})

maf_df

In [ ]:
coding_an = an.filter(pl.col('id').is_in(coding_vars)).drop(['col', 'af_ukb', 'afbeta_ukb']).collect()

coding_an = maf_df.join(coding_an, on='id', how='inner')
coding_an

In [ ]:
maf = 0.001
coding_an.filter((pl.col('region')=='ENSG00000106633') & (pl.col('af_ukb') < maf))

In [ ]:
sel_ids = coding_an.filter((pl.col('region')=='ENSG00000106633') & (pl.col('af_ukb') < maf)).select('id')['id']

vm_og.filter(
    pl.col('id').is_in(sel_ids)
)

In [ ]:
vm_og.filter(pl.col('id')=="chr7:44147830:G:A")

In [ ]:
vm.write_parquet("/home/dnanexus/data_dir/dms_coding.ag/variant_metadata.parquet")
coding_an.write_parquet("/home/dnanexus/data_dir/dms_coding.ag/annotations.parquet")

### Write samples zarr array

In [ ]:
import zarr

old_store = zarr.open('/home/dnanexus/data_dir/dms_anngeno.ag/zarr_store', mode='r')
samps = old_store['samples']
samps

In [ ]:
# Create a new Zarr v3 store
new_path = zarr_dir = "/home/dnanexus/data_dir/samples_zarr"
new_store = zarr.open(
    new_path,
    mode='w',
    zarr_version=3
)

# Copy the array into the new store
new_store.create_dataset(
    name='samples',
    shape=samps.shape,
    dtype=samps.dtype,
    chunks=samps.chunks,
    data=samps[...],   # load data into memory
    overwrite=True
)

### Check new anngeno

In [ ]:
ag_new = AnnGeno(
    "/home/dnanexus/data_dir/dms_coding.ag",
    low_mem = True
)

ag_new

In [ ]:
%%time

ag_new.get_region("ENSG00000122299")

In [ ]:
%%time

ag_new.get_many_regions(["ENSG00000122299","ENSG00000073584","ENSG00000009709"])